In [3]:
# -*- coding: utf-8 -*-
"""
Add RQ1_Adopted (True/False) to the main dataset.

RQ1 definition:
  RQ1_Adopted := (AT == 1) AND (B == 1 OR Y == 1)
  where:
    AT = Instru_test
    B  = build_instru_signal
    Y  = ci_android_signal

Reads from:
  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet

Outputs:
  Overwrites the original input file (CSV/XLSX).
"""

from pathlib import Path
import pandas as pd
import re

# ------------------- CONFIG -------------------
DATA_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet"
# Prefer this file name if present; otherwise the script will pick the first CSV/XLSX it finds:
PREFERRED_FILE = "3.2_Total_Repo.csv"

# Column names (change here if your dataset uses different names)
COL_AT = "Instru_test"
COL_B  = "instru_t_signal_config"
COL_Y  = "ci_android_signal"

# New column name to add:
RQ1_COL = "RQ1_F1"
# ----------------------------------------------


def find_input_file(dir_path: str, preferred: str = None) -> Path:
    p = Path(dir_path)
    if preferred and (p / preferred).exists():
        return p / preferred
    # Fallbacks: first CSV then first Excel
    for pattern in ("*.csv", "*.xlsx", "*.xls"):
        files = sorted(p.glob(pattern))
        if files:
            return files[0]
    raise FileNotFoundError(f"No CSV/XLSX file found in {dir_path!r}.")


def robust_to01(series: pd.Series) -> pd.Series:
    """Coerce booleans/strings/numbers like '1', '1.0', 'true', 'yes' -> 1; else 0."""
    s = series.copy()

    # Already boolean?
    if s.dtype == bool:
        return s.astype(int)

    # Numeric? threshold at > 0
    if pd.api.types.is_numeric_dtype(s):
        return (s.fillna(0).astype(float) > 0).astype(int)

    # String-like: normalize tokens
    t = s.astype(str).str.strip().str.lower()
    token_map = {
        "1":1, "true":1, "yes":1, "y":1, "t":1, "on":1, "present":1,
        "0":0, "false":0, "no":0, "n":0, "f":0, "off":0, "":0, "none":0, "null":0, "na":0, "nan":0
    }
    mapped = t.map(token_map)

    # Numeric-looking strings like "1.0", "0.0", "+2", etc.
    numeric_like = pd.to_numeric(t.str.replace(r"[^0-9\.\-]+", "", regex=True), errors="coerce")
    num01 = (numeric_like.fillna(0).astype(float) > 0).astype(int)

    out = mapped.where(mapped.notna(), num01).fillna(0).astype(int)
    return out


def main():
    in_path = find_input_file(DATA_DIR, PREFERRED_FILE)
    print(f"Reading: {in_path}")

    # Load file
    if in_path.suffix.lower() in (".xlsx", ".xls"):
        df = pd.read_excel(in_path)
        filetype = "excel"
    else:
        df = pd.read_csv(in_path, encoding="utf-8-sig")
        filetype = "csv"

    # Sanity check columns exist (with graceful message)
    missing = [c for c in (COL_AT, COL_B, COL_Y) if c not in df.columns]
    if missing:
        print("\n⚠️  The following expected columns were not found exactly as named:")
        for c in missing:
            print(f"   - {c}")
        print("\nAvailable columns are:")
        print(list(df.columns))
        print("\nIf your dataset uses slightly different names, edit COL_AT/COL_B/COL_Y at the top and re-run.")
        # Attempt a soft match by normalized names to help recover from minor header differences
        def clean_name(name: str) -> str:
            n = str(name).strip().lower()
            n = re.sub(r"[\s\-\./]+", "_", n)
            n = re.sub(r"[^a-z0-9_]+", "", n)
            n = re.sub(r"_+", "_", n).strip("_")
            return n
        norm_lookup = {clean_name(c): c for c in df.columns}
        wanted = {COL_AT: None, COL_B: None, COL_Y: None}
        for logical, target in [(COL_AT, "instru_test"), (COL_B, "build_instru_signal"), (COL_Y, "ci_android_signal")]:
            key = clean_name(logical)
            if key in norm_lookup:
                wanted[logical] = norm_lookup[key]
        if any(v is None for v in wanted.values()):
            print("\n🔎 Could not safely map all three columns automatically—aborting to avoid wrong results.")
            return
        else:
            print("\n✅ Soft-mapped columns by normalized names:")
            for k,v in wanted.items():
                print(f"   {k} -> {v}")
            at_col, b_col, y_col = wanted[COL_AT], wanted[COL_B], wanted[COL_Y]
    else:
        at_col, b_col, y_col = COL_AT, COL_B, COL_Y

    # Normalize to 0/1
    AT = robust_to01(df[at_col])
    B  = robust_to01(df[b_col])
    Y  = robust_to01(df[y_col])

    # RQ1 flag (boolean)
    rq1_flag = (AT.eq(1) & (B.eq(1) | Y.eq(1)))
    df[RQ1_COL] = rq1_flag  # True/False

    # Optional: also add a 0/1 version if you like
    df[RQ1_COL + "_int"] = rq1_flag.astype(int)

    # Quick summary
    total = len(df)
    adopters = int(rq1_flag.sum())
    rate = adopters / total * 100 if total else 0.0
    print(f"\nAdded column: {RQ1_COL}")
    print(f"Adopters by RQ1: {adopters} of {total}  ({rate:.1f}%)")

    # --------- SAVE: OVERWRITE THE INPUT FILE ----------
    if filetype == "excel":
        # Overwrite with a single sheet named 'data' (same behavior as before)
        with pd.ExcelWriter(in_path, engine="xlsxwriter") as xw:
            df.to_excel(xw, index=False, sheet_name="data")
    else:
        # Keep UTF-8 with BOM to mirror the read encoding
        df.to_csv(in_path, index=False, encoding="utf-8-sig")

    print(f"[OK] Overwrote: {in_path}")

if __name__ == "__main__":
    main()


Reading: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv

Added column: RQ1_F1
Adopters by RQ1: 1542 of 4518  (34.1%)
[OK] Overwrote: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv


In [6]:
# -*- coding: utf-8 -*-
"""
Create RQ1_F2 categories:
- Fully_Auto: Instru_test=1 AND instru_t_signal_config=1 AND ci_android_signal=1
- Semi_Auto:  Instru_test=1 AND instru_t_signal_config=1 AND ci_android_signal=0
- CI_Only:    Instru_test=1 AND instru_t_signal_config=0 AND ci_android_signal=1
- Others:     All remaining cases (incl. no test code, or test code with no signals)

Input:
  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo_with_RQ1.csv

Output:
  Overwrites the original CSV in place.
"""

from pathlib import Path
import numpy as np
import pandas as pd

# ---- path to your file ----
CSV_PATH = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv"
csv_path = Path(CSV_PATH)

# ---- load ----
df = pd.read_csv(csv_path, encoding="utf-8-sig")

# ---- required columns (exact names as used in your dataset) ----
required = ["Instru_test", "instru_t_signal_config", "ci_android_signal"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing required column(s): {missing}. "
                   f"Expected exactly: {required}")

# ---- normalize to 0/1 (handles bools, numbers, strings) ----
def to01(s: pd.Series) -> pd.Series:
    if s.dtype == bool:
        return s.astype(int)
    if pd.api.types.is_numeric_dtype(s):
        return (pd.to_numeric(s, errors="coerce").fillna(0) > 0).astype(int)
    t = s.astype(str).str.strip().str.lower()
    token_map = {
        "1":1,"true":1,"yes":1,"y":1,"t":1,"on":1,
        "0":0,"false":0,"no":0,"n":0,"f":0,"off":0,"":0,"none":0,"null":0,"na":0,"nan":0
    }
    mapped = t.map(token_map)
    num = pd.to_numeric(t.str.replace(r"[^0-9\.\-]+", "", regex=True), errors="coerce")
    num01 = (num.fillna(0) > 0).astype(int)
    return mapped.where(mapped.notna(), num01).astype(int)

AT = to01(df["Instru_test"])
B  = to01(df["instru_t_signal_config"])
Y  = to01(df["ci_android_signal"])

# ---- build category ----
conditions = [
    (AT.eq(1) & B.eq(1) & Y.eq(1)),  # Fully automated
    (AT.eq(1) & B.eq(1) & Y.eq(0)),  # Semi automated
    (AT.eq(1) & B.eq(0) & Y.eq(1)),  # CI-only
]
choices = ["Fully_Auto", "Semi_Auto", "CI_Only"]

df["RQ1_F2"] = np.select(conditions, choices, default="Others")
df["RQ1_F2"] = pd.Categorical(df["RQ1_F2"],
                              categories=["Fully_Auto", "Semi_Auto", "CI_Only", "Others"],
                              ordered=True)

# ---- optional: quick sanity printouts ----
print("\nCounts by RQ1_F2:")
print(df["RQ1_F2"].value_counts(dropna=False))

# If RQ1 flag exists, show distribution within adopters
rq1_col = None
for cand in ["RQ1_Adopted_int", "RQ1_Adopted", "RQ1", "rq1"]:
    if cand in df.columns:
        rq1_col = cand
        break

if rq1_col:
    rq1 = to01(df[rq1_col])
    print("\nCounts within adopters (RQ1==1):")
    print(df.loc[rq1.eq(1), "RQ1_F2"].value_counts())
    print("\nShares within adopters (RQ1==1):")
    print((df.loc[rq1.eq(1), "RQ1_F2"].value_counts(normalize=True) * 100).round(1).astype(str) + "%")

# ---- save (OVERWRITE INPUT) ----
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"\n[OK] Overwrote in place: {csv_path}")



Counts by RQ1_F2:
RQ1_F2
Others        2976
Semi_Auto     1177
Fully_Auto     335
CI_Only         30
Name: count, dtype: int64

[OK] Overwrote in place: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\1_Main_Spreadsheet\3.2_Total_Repo.csv


Create the execution env and test invocation columns

In [1]:
# -*- coding: utf-8 -*-
"""
From confidence_reason, derive:
  execution_environment ∈ {Emulator, Third Party, GMD, Real Device, Unknown}
  test_invocation       ∈ {Gradle, ADB, 3P CLIs, Unknown}

Input (preferred):
  C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\3.2_Total_Repo_with_RQ1_F2.csv

Output:
  3.2_Total_Repo_with_RQ1_F2_with_styles.csv  (same folder)
"""

import re
from pathlib import Path
import pandas as pd
import numpy as np

# -------------------------------------------------------------------
# 1) Load
# -------------------------------------------------------------------
PREFERRED = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\3.2_Total_Repo_with_RQ1_F2.csv")
FALLBACK  = Path("/mnt/data/3.2_Total_Repo_with_RQ1_F2.csv")

if PREFERRED.exists():
    csv_path = PREFERRED
elif FALLBACK.exists():
    csv_path = FALLBACK
else:
    raise FileNotFoundError("Dataset not found in either preferred or fallback path. Update 'PREFERRED' to your actual file.")

df = pd.read_csv(csv_path, encoding="utf-8-sig")

if "confidence_reason" not in df.columns:
    raise KeyError("Expected column 'confidence_reason' not found in dataset.")

cr = df["confidence_reason"].fillna("").astype(str)

# Normalize (lowercase for regex)
cr_low = cr.str.lower()

# -------------------------------------------------------------------
# 2) Heuristics (regex) for environment & invocation
# -------------------------------------------------------------------
# --- Environment: Third Party (e.g., Firebase Test Lab) ---
pat_env_third_party = re.compile(r"\bgcloud firebase\b|firebase test lab|\bftl\b|device farm|sauce\s*labs|browserstack", re.IGNORECASE)

# --- Environment: Gradle Managed Devices (GMD) ---
pat_env_gmd = re.compile(
    r"managed\s*devices?|managedvirtualdevice|testoptions\.manageddevices|alldevices.*androidtest|device\s*androidtest",
    re.IGNORECASE
)

# --- Environment: Emulator (generic emulator/AVD setup tokens) ---
pat_env_emulator = re.compile(
    r"\bemulator\b|android-?emulator-?runner|reactivecircus|avd(manager)?|create avd|sdkmanager .*system-images|qemu",
    re.IGNORECASE
)

# --- Environment: Real Device ---
# If we see "adb -s emulator-" it's still emulator; prefer emulator detection above.
# Treat explicit "physical", "real device", or USB-only references as Real Device.
pat_env_real = re.compile(
    r"\breal\s*device\b|\bphysical\b|usb\s*device|udid\b",
    re.IGNORECASE
)

# --- Invocation: Gradle (connected/managed tasks) ---
pat_inv_gradle = re.compile(
    r"gradle|gradlew|connectedandroidtest|connected.*androidtest|connectedcheck|assemble\w*androidtest|device\s*androidtest|alldevices.*androidtest|managed",
    re.IGNORECASE
)

# --- Invocation: ADB / am instrument ---
pat_inv_adb = re.compile(
    r"\badb\b.*(shell|install|uninstall|logcat)|\bam instrument\b",
    re.IGNORECASE
)

# --- Invocation: Third-Party CLIs ---
pat_inv_3p = re.compile(
    r"\bgcloud firebase\b|firebase test lab|\bftl\b|device farm|sauce\s*labs|browserstack|bitrise run|circleci local execute",
    re.IGNORECASE
)

def classify_environment(text: str) -> str:
    t = text  # already lowercased upstream
    # Order matters: FTL (3P) > GMD > Emulator > Real
    if pat_env_third_party.search(t):
        return "Third Party"
    if pat_env_gmd.search(t):
        return "GMD"
    if pat_env_emulator.search(t):
        return "Emulator"
    # If ADB mentions emulator explicitly, treat as Emulator; else Real Device
    if "adb" in t:
        if "emulator-" in t or re.search(r"\bemulator\b", t):
            return "Emulator"
        # explicit USB/physical tokens push to Real Device
        if pat_env_real.search(t):
            return "Real Device"
    if pat_env_real.search(t):
        return "Real Device"
    return "Unknown"

def classify_invocation(text: str) -> str:
    t = text  # already lowercased
    if pat_inv_3p.search(t):
        return "3P CLIs"
    if pat_inv_adb.search(t):
        return "ADB"
    if pat_inv_gradle.search(t):
        return "Gradle"
    return "Unknown"

# -------------------------------------------------------------------
# 3) Apply classifiers
# -------------------------------------------------------------------
df["execution_environment"] = cr_low.apply(classify_environment)
df["test_invocation"]       = cr_low.apply(classify_invocation)

# Optional: make them categorical for clean summaries
df["execution_environment"] = pd.Categorical(
    df["execution_environment"],
    categories=["Emulator", "Third Party", "GMD", "Real Device", "Unknown"],
    ordered=False
)
df["test_invocation"] = pd.Categorical(
    df["test_invocation"],
    categories=["Gradle", "ADB", "3P CLIs", "Unknown"],
    ordered=False
)

# -------------------------------------------------------------------
# 4) Quick sanity summaries (print to console)
# -------------------------------------------------------------------
print("\nExecution environment (overall):")
print(df["execution_environment"].value_counts(dropna=False))

print("\nTest invocation (overall):")
print(df["test_invocation"].value_counts(dropna=False))

# If you want it within adopters only:
adopt_col = None
for cand in ["RQ1_Adopted_int","RQ1_Adopted","RQ1","rq1"]:
    if cand in df.columns:
        adopt_col = cand
        break

if adopt_col is not None:
    adopters = df[(pd.to_numeric(df[adopt_col], errors="coerce").fillna(0) > 0)]
    print(f"\nWithin adopters (N={len(adopters)}):")
    print("  Execution environment:")
    print(adopters["execution_environment"].value_counts(dropna=False))
    print("\n  Test invocation:")
    print(adopters["test_invocation"].value_counts(dropna=False))

# -------------------------------------------------------------------
# 5) Save
# -------------------------------------------------------------------
out_path = csv_path.with_name(csv_path.stem + "_with_styles.csv")
df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\nSaved: {out_path}")



Execution environment (overall):
execution_environment
Unknown        4210
GMD             161
Emulator        121
Third Party      26
Real Device       0
Name: count, dtype: int64

Test invocation (overall):
test_invocation
Unknown    4125
Gradle      353
3P CLIs      26
ADB          14
Name: count, dtype: int64

Within adopters (N=1542):
  Execution environment:
execution_environment
Unknown        1283
GMD             136
Emulator         98
Third Party      25
Real Device       0
Name: count, dtype: int64

  Test invocation:
test_invocation
Unknown    1214
Gradle      294
3P CLIs      25
ADB           9
Name: count, dtype: int64

Saved: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\RQ1\3.2_Total_Repo_with_RQ1_F2_with_styles.csv
